In [9]:
DATASET = 'cora'  # 'cora' or 'citeseer'


In [10]:
import os, re, statistics

STATS_RE = re.compile(r'Test accuracy:\s+([\d.]+)\s+\+/-\s+([\d.]+)')
BEST_RE  = re.compile(r'Best test ACC:\s+([\d.]+)')

def parse_log(path):
    try:
        txt = open(path).read()
    except Exception:
        return None, None
    m = STATS_RE.search(txt)
    if m:
        mean, std = float(m.group(1)), float(m.group(2))
        if mean <= 1.0: mean *= 100; std *= 100
        return round(mean, 2), round(std, 2)
    accs = [float(x) for x in BEST_RE.findall(txt)]
    if accs:
        if accs[0] <= 1.0: accs = [a * 100 for a in accs]
        return round(statistics.mean(accs), 2), round(statistics.pstdev(accs), 2)
    return None, None

PAPER = {'cora': (84.2, 0.5), 'citeseer': (73.5, 0.6)}
paper_mean, paper_std = PAPER[DATASET]
print(f'Results for {DATASET} (paper baseline: {paper_mean} +/- {paper_std})')

for experiment in ['temperature', 'c', 'joint', 'lr']:
    log_dir = os.path.join('logs', 'sweep', DATASET, experiment)
    print(log_dir)
    if not os.path.exists(log_dir):
        print(f'[{experiment}] No logs found yet')
        continue
    print(f'\n{"="*55}')
    print(f'  {experiment}')
    print(f'{"="*55}')
    rows = []
    for fname in sorted(os.listdir(log_dir)):
        if not fname.endswith('.log'): continue
        mean, std = parse_log(os.path.join(log_dir, fname))
        rows.append((mean or 0, std or 0, fname.replace('.log', '')))
    rows.sort(reverse=True)
    for mean, std, tag in rows:
        delta  = mean - paper_mean
        marker = ' <- best' if mean == rows[0][0] else ''
        print(f'  {tag:<40s}  {mean:.2f} +/- {std:.2f}  '
              f'({("+" if delta>=0 else "")}{delta:.1f}){marker}')

print('\nFill BEST_TEMPERATURE and BEST_C into Experiment 4 based on the joint results.')

Results for cora (paper baseline: 84.2 +/- 0.5)
logs/sweep/cora/temperature

  temperature
  temperature0.2_c0                         83.27 +/- 0.45  (-0.9) <- best
  temperature0.5_c0                         82.83 +/- 0.49  (-1.4)
  temperature0.1_c0                         82.43 +/- 0.45  (-1.8)
  temperature1.0_c0                         81.70 +/- 0.43  (-2.5)
  temperature0.05_c0                        81.47 +/- 1.14  (-2.7)
logs/sweep/cora/c

  c
  temperature0.2_c1                         83.27 +/- 0.45  (-0.9) <- best
  temperature0.2_c0                         83.27 +/- 0.45  (-0.9) <- best
  temperature0.2_c10                        82.93 +/- 0.91  (-1.3)
  temperature0.2_c25                        82.93 +/- 0.12  (-1.3)
  temperature0.2_c5                         82.73 +/- 0.25  (-1.5)
  temperature0.2_c100                       82.57 +/- 0.61  (-1.6)
  temperature0.2_c50                        82.53 +/- 0.69  (-1.7)
logs/sweep/cora/joint

  joint
  temperature0.2_c0        